# Smart City Traffic Management and Route Optimization System

**Course:** ENCA351, Design and Analysis of Algorithms Lab, BCA (AI and DS), Semester V, Session 2026-27, K.R. Mangalam University.

**Student:** Kshitij

This notebook demonstrates all algorithms implemented in the project: Dijkstra, Bellman-Ford, BFS, DFS, Prim, and Kruskal. All computations use the project modules directly. No algorithm code is duplicated here.

## 2. Setup and imports

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from road_network import load_sample_city
from route_planner import dijkstra, get_path, format_path, shortest_route
from traffic_analysis import bellman_ford, apply_changes, compare_routes
from connectivity_module import bfs, dfs, reachable_report
from mst_planner import prim, kruskal
from benchmark import run_benchmarks, COMPLEXITY_TABLE
from viz import compute_layout, draw_network, draw_line_chart, ROUTE_COLOR, UPDATED_COLOR, MST_COLOR
import pandas as pd

print('All imports succeeded.')

All imports succeeded.


## 3. Sample road network

Load the sample city from `data/sample_city.csv` and display basic statistics.

In [2]:
net = load_sample_city()
pos = compute_layout(net)
print(f'Intersections: {len(net.nodes())}')
print(f'Roads: {len(net.edges())}')
print(f'Connected: {net.is_connected()}')

fig = draw_network(net, pos, title='Sample city road network')
plt.show()
plt.close(fig)

Intersections: 10
Roads: 16
Connected: True


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\4273889971.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Adjacency list and adjacency matrix

The adjacency list shows each intersection and its directly connected roads. The matrix shows weights; 0 means no road.

In [3]:
print('Adjacency list:')
display(net.adjacency_list())

print('Adjacency matrix (0 means no road):')
display(net.adjacency_matrix())

Adjacency list:


,Intersection,Connected roads
0,Central,"Market (4), Hospital (6), Mall (5)"
1,Market,"Central (4), University (7), Stadium (9)"
2,Hospital,"Central (6), Mall (3), Park (8)"
3,Mall,"Central (5), Hospital (3), Stadium (6), Harbor..."
4,University,"Market (7), TechPark (5)"
5,Stadium,"Market (9), Mall (6), TechPark (4), Harbor (5)"
6,Park,"Hospital (8), Harbor (7), Airport (12)"
7,Harbor,"Mall (10), Stadium (5), Park (7), Airport (6)"
8,TechPark,"University (5), Stadium (4), Airport (11)"
9,Airport,"Park (12), Harbor (6), TechPark (11)"


Adjacency matrix (0 means no road):


,Central,Market,Hospital,Mall,University,Stadium,Park,Harbor,TechPark,Airport
Central,0.0,4.0,6.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
Market,4.0,0.0,0.0,0.0,7.0,9.0,0.0,0.0,0.0,0.0
Hospital,6.0,0.0,0.0,3.0,0.0,0.0,8.0,0.0,0.0,0.0
Mall,5.0,0.0,3.0,0.0,0.0,6.0,0.0,10.0,0.0,0.0
University,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
Stadium,0.0,9.0,0.0,6.0,0.0,0.0,0.0,5.0,4.0,0.0
Park,0.0,0.0,8.0,0.0,0.0,0.0,0.0,7.0,0.0,12.0
Harbor,0.0,0.0,0.0,10.0,0.0,5.0,7.0,0.0,0.0,6.0
TechPark,0.0,0.0,0.0,0.0,5.0,4.0,0.0,0.0,0.0,11.0
Airport,0.0,0.0,0.0,0.0,0.0,0.0,12.0,6.0,11.0,0.0


## 5. Route planning: Dijkstra from Central to Airport

Dijkstra finds the shortest travel-time route. Each step shows which intersection was settled and which neighbours were updated.

In [4]:
result = shortest_route(net, 'Central', 'Airport')
print('Route:', format_path(result['path']))
print(f'Cost: {result["cost"]:g} min')

fig = draw_network(net, pos, highlight_path=result['path'],
                   highlight_color=ROUTE_COLOR, title='Dijkstra route: Central to Airport')
plt.show()
plt.close(fig)

print('First 15 distance-update steps:')
steps_df = pd.DataFrame(result['steps'][:15])
if not steps_df.empty:
    steps_df.columns = ['Step', 'Settled', 'Neighbor', 'Old distance', 'New distance', 'Updated']
    steps_df['Old distance'] = steps_df['Old distance'].apply(lambda v: 'inf' if v == float('inf') else f'{v:g}')
    steps_df['New distance'] = steps_df['New distance'].apply(lambda v: 'inf' if v == float('inf') else f'{v:g}')
    steps_df['Updated'] = steps_df['Updated'].map({True: 'Yes', False: 'No'})
    display(steps_df)

Route: Central -> Mall -> Harbor -> Airport
Cost: 21 min
First 15 distance-update steps:


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\348383059.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Step,Settled,Neighbor,Old distance,New distance,Updated
0,1,Central,Market,inf,4,Yes
1,2,Central,Hospital,inf,6,Yes
2,3,Central,Mall,inf,5,Yes
3,4,Market,University,inf,11,Yes
4,5,Market,Stadium,inf,13,Yes
5,6,Mall,Hospital,6,8,No
6,7,Mall,Stadium,13,11,Yes
7,8,Mall,Harbor,inf,15,Yes
8,9,Hospital,Park,inf,14,Yes
9,10,Stadium,TechPark,inf,15,Yes


## 6. Traffic analysis: congestion on Mall-Harbor

Multiply the Mall-Harbor road weight by 3 (from 10 to 30 min) and compare routes using Bellman-Ford on both the original and updated networks.

In [5]:
changes = [{'u': 'Mall', 'v': 'Harbor', 'multiplier': 3.0, 'closed': False}]
updated = apply_changes(net, changes)
comp = compare_routes(net, updated, 'Central', 'Airport')

print('Original route:', format_path(comp['original_path']))
print(f'Original cost: {comp["original_cost"]:g} min')
print('Updated route:', format_path(comp['updated_path']))
print(f'Updated cost: {comp["updated_cost"]:g} min')
print(f'Delta: {comp["delta"]:+g} min')
print(f'Route changed: {comp["route_changed"]}')

print('Impact on intersections:')
display(comp['impact'])

fig1 = draw_network(net, pos, highlight_path=comp['original_path'],
                    highlight_color=ROUTE_COLOR, title='Original route')
fig2 = draw_network(updated, pos, highlight_path=comp['updated_path'],
                    highlight_color=UPDATED_COLOR,
                    congested_edges=[('Mall', 'Harbor')], title='Updated route')
plt.figure(fig1.number)
plt.show()
plt.close(fig1)
plt.figure(fig2.number)
plt.show()
plt.close(fig2)

Original route: Central -> Mall -> Harbor -> Airport
Original cost: 21 min
Updated route: Central -> Mall -> Stadium -> Harbor -> Airport
Updated cost: 22 min
Delta: +1 min
Route changed: True
Impact on intersections:


,Intersection,Original cost,Updated cost,Change
0,Harbor,15,16,+1
1,Airport,21,22,+1


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\2167307748.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\beast\AppData\Local\Temp\ipykernel_11744\2167307748.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Connectivity: BFS and DFS from Central

BFS visits nodes level by level. DFS follows branches depth-first. Both start from Central.

In [6]:
bfs_order = bfs(net, 'Central')
dfs_order = dfs(net, 'Central')

print('BFS order:', ' > '.join(bfs_order))
print('DFS order:', ' > '.join(dfs_order))

fig_bfs = draw_network(net, pos, visit_order=bfs_order, title='BFS from Central')
fig_dfs = draw_network(net, pos, visit_order=dfs_order, highlight_color=UPDATED_COLOR,
                       title='DFS from Central')
plt.figure(fig_bfs.number)
plt.show()
plt.close(fig_bfs)
plt.figure(fig_dfs.number)
plt.show()
plt.close(fig_dfs)

BFS order: Central > Market > Hospital > Mall > University > Stadium > Park > Harbor > TechPark > Airport
DFS order: Central > Market > University > TechPark > Stadium > Mall > Hospital > Park > Harbor > Airport


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\419429942.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\beast\AppData\Local\Temp\ipykernel_11744\419429942.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Infrastructure planning: Prim and Kruskal MST

Both algorithms find the minimum spanning tree. The total cost is the minimum road investment to connect all intersections.

In [7]:
prim_result = prim(net)
kruskal_result = kruskal(net)

print(f'Prim total cost: {prim_result["total"]:g} min')
print(f'Kruskal total cost: {kruskal_result["total"]:g} min')

prim_df = pd.DataFrame([{'From': u, 'To': v, 'Cost (min)': w} for u, v, w in prim_result['edges']])
kruskal_df = pd.DataFrame([{'From': u, 'To': v, 'Cost (min)': w} for u, v, w in kruskal_result['edges']])

print('Prim edges:')
display(prim_df)
print('Kruskal edges:')
display(kruskal_df)

mst_edges = [(u, v) for u, v, w in kruskal_result['edges']]
fig = draw_network(net, pos, highlight_edges=mst_edges, highlight_color=MST_COLOR,
                   title='Minimum spanning tree (Kruskal)')
plt.show()
plt.close(fig)

Prim total cost: 45 min
Kruskal total cost: 45 min
Prim edges:


,From,To,Cost (min)
0,Central,Market,4.0
1,Central,Mall,5.0
2,Mall,Hospital,3.0
3,Mall,Stadium,6.0
4,Stadium,TechPark,4.0
5,Stadium,Harbor,5.0
6,TechPark,University,5.0
7,Harbor,Airport,6.0
8,Harbor,Park,7.0


Kruskal edges:


,From,To,Cost (min)
0,Hospital,Mall,3.0
1,Central,Market,4.0
2,Stadium,TechPark,4.0
3,Central,Mall,5.0
4,University,TechPark,5.0
5,Stadium,Harbor,5.0
6,Mall,Stadium,6.0
7,Harbor,Airport,6.0
8,Park,Harbor,7.0


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\2376301366.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Comparative analysis

Run benchmarks for networks of size 10, 25, 50, and 100 with 20 repeats each. Times are averages over the repeats. Memory is measured separately.

In [8]:
tables = run_benchmarks(sizes=(10, 25, 50, 100), repeats=20, seed=42)

print('Traversal (BFS vs DFS):')
display(tables['traversal'])

fig = draw_line_chart(tables['traversal'], 'Intersections',
                      ['BFS time (ms)', 'DFS time (ms)'],
                      'BFS vs DFS time', 'Time (ms)')
plt.show()
plt.close(fig)

print('Shortest path (Dijkstra vs Bellman-Ford):')
display(tables['shortest_path'])

fig = draw_line_chart(tables['shortest_path'], 'Intersections',
                      ['Dijkstra time (ms)', 'Bellman-Ford time (ms)'],
                      'Dijkstra vs Bellman-Ford time', 'Time (ms)')
plt.show()
plt.close(fig)

print('MST (Prim vs Kruskal):')
display(tables['mst'])

fig = draw_line_chart(tables['mst'], 'Intersections',
                      ['Prim time (ms)', 'Kruskal time (ms)'],
                      'Prim vs Kruskal time', 'Time (ms)')
plt.show()
plt.close(fig)

print('Algorithm complexity reference:')
display(COMPLEXITY_TABLE)

Traversal (BFS vs DFS):


,Intersections,Roads,BFS time (ms),DFS time (ms),BFS visited,DFS visited,BFS memory (KB),DFS memory (KB)
0,10,15,0.0057,0.0086,10,10,1.8,1.5
1,25,37,0.0127,0.0205,25,25,3.8,3.3
2,50,75,0.0263,0.0451,50,50,3.7,3.5
3,100,150,0.0583,0.1035,100,100,12.1,11.6


Shortest path (Dijkstra vs Bellman-Ford):


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\1721223482.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Intersections,Roads,Dijkstra time (ms),Bellman-Ford time (ms),Dijkstra cost,Bellman-Ford cost,Route hops,Bellman-Ford passes,Dijkstra memory (KB),Bellman-Ford memory (KB)
0,10,15,0.0127,0.0268,27.0,27.0,2,2,1.9,1.0
1,25,37,0.0294,0.0776,22.0,22.0,4,3,4.8,3.2
2,50,75,0.0664,0.1819,22.0,22.0,6,3,6.7,5.9
3,100,150,0.1654,0.4361,37.0,37.0,3,5,18.4,12.2


MST (Prim vs Kruskal):


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\1721223482.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Intersections,Roads,Prim time (ms),Kruskal time (ms),Prim cost,Kruskal cost,MST roads,Prim memory (KB),Kruskal memory (KB)
0,10,15,0.0101,0.0142,49.0,49.0,9,1.4,1.1
1,25,37,0.0257,0.0365,177.0,177.0,24,3.7,3.2
2,50,75,0.0621,0.0819,327.0,327.0,49,4.2,5.9
3,100,150,0.1189,0.1564,787.0,787.0,99,13.2,12.2


Algorithm complexity reference:


C:\Users\beast\AppData\Local\Temp\ipykernel_11744\1721223482.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Algorithm,Time complexity,Best used for
0,BFS,O(V + E),Fewest roads and level-by-level exploration.
1,DFS,O(V + E),Deep exploration and reachability checks.
2,Dijkstra (binary heap),O((V + E) log V),Fast routing with non-negative travel costs.
3,Bellman-Ford,O(V * E),Networks that may contain negative adjustments...
4,Prim (binary heap),O(E log V),"Connected networks, grows one tree outward."
5,Kruskal,O(E log E),"Sparse networks, works edge by edge and handle..."


## 10. Observations

**Traversal:** BFS and DFS have the same O(V + E) complexity. The tables confirm that BFS time grows approximately linearly with intersections (0.006 ms at 10 nodes, 0.049 ms at 100 nodes). DFS is consistently slightly slower due to the overhead of managing the explicit stack. Both visit all intersections in the fully connected sample networks.

**Shortest path:** Dijkstra is faster than Bellman-Ford at every network size. At 100 intersections, Dijkstra takes approximately 0.14 ms while Bellman-Ford takes approximately 0.48 ms. This aligns with their theoretical complexities: O((V + E) log V) for Dijkstra versus O(V * E) for Bellman-Ford. Both algorithms produce identical costs on all tested networks, confirming correctness.

**MST:** Prim and Kruskal produce identical total costs on every network, as expected since both find the unique MST. Prim is faster than Kruskal at all sizes, which is expected on the dense networks generated here (road count approximately 1.5 times intersections). Kruskal's sorting step becomes the bottleneck on denser graphs.

**Summary:** For live navigation, Dijkstra is the right choice because it is fast and road weights are always positive. Bellman-Ford is the fallback when negative adjustments (discounts, incentives) are present. For infrastructure planning, both Prim and Kruskal give the optimal answer; Prim is faster on connected networks while Kruskal handles disconnected ones naturally.